# Introduction

This notebook demonstrates how to train custom openWakeWord models using pre-defined datasets and an automated process for dataset generation and training. While not guaranteed to always produce the best performing model, the methods shown in this notebook often produce baseline models with releatively strong performance.

Manual data preparation and model training (e.g., see the [training models](training_models.ipynb) notebook) remains an option for when full control over the model development process is needed.

At a high level, the automatic training process takes advantages of several techniques to try and produce a good model, including:

- Early-stopping and checkpoint averaging (similar to [stochastic weight averaging](https://arxiv.org/abs/1803.05407)) to search for the best models found during training, according to the validation data
- Variable learning rates with cosine decay and multiple cycles
- Adaptive batch construction to focus on only high-loss examples when the model begins to converge, combined with gradient accumulation to ensure that batch sizes are still large enough for stable training
- Cycical weight schedules for negative examples to help the model reduce false-positive rates

See the contents of the `train.py` file for more details.

# Environment Setup

To begin, we'll need to install the requirements for training custom models. In particular, a relatively recent version of Pytorch and custom fork of the [piper-sample-generator](https://github.com/dscripka/piper-sample-generator) library for generating synthetic examples for the custom model.

**Important Note!** Currently, automated model training is only supported on linux systems due to the requirements of the text to speech library used for synthetic sample generation (Piper). It may be possible to use Piper on Windows/Mac systems, but that has not (yet) been tested.

In [ ]:
## Environment setup

# install piper-sample-generator (currently only supports linux systems)
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!cd ~/openwakeword_project/piper-sample-generator && git checkout 213d4d5	

!pip install piper-phonemize
!pip install webrtcvad

# install openwakeword (full installation to support training)
# INSTALL OPENWEAKEWORD ()
!git clone https://github.com/dscripka/openwakeword 
!pip install -e ./openwakeword[full]
!cd openwakeword

# install other dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install tensorflow-cpu==2.8.1
!pip install tensorflow_probability==0.16.0
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19

# Download required models (workaround for Colab)
import os
os.makedirs("./openwakeword/openwakeword/resources/models")
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite


In [1]:
# Imports

import os
import sys
from pathlib import Path
import torch
import datasets
import numpy as np
import yaml
import shutil
from tqdm import tqdm
import scipy
import random
import math
from glob import glob
import uuid
import warnings


# Download Data

When training new openWakeWord models using the automated procedure, four specific types of data are required:

1) Synthetic examples of the target word/phrase generated with text-to-speech models

2) Synthetic examples of adversarial words/phrases generated with text-to-speech models

3) Room impulse reponses and noise/background audio data to augment the synthetic examples and make them more realistic

4) Generic "negative" audio data that is very unlikely to contain examples of the target word/phrase in the context where the model should detect it. This data can be the original audio data, or precomputed openWakeWord features ready for model training.

5) Validation data to use for early-stopping when training the model.

For the purposes of this notebook, all five of these sources will either be generated manually or can be obtained from HuggingFace thanks to their excellent `datasets` library and extremely generous hosting policy. Also note that while only a portion of some datasets are downloaded, for the best possible performance it is recommended to download the entire dataset and keep a local copy for future training runs.

In [ ]:
# Download pre-computed openWakeWord features for training and validation

# training set (~2,000 hours from the ACAV100M Dataset)
# See https://huggingface.co/datasets/davidscripka/openwakeword_features for more information
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy

# validation set for false positive rate estimation (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

## Conversione e Pre-calcolo Feature – Common Voice (negativi diretti)

Questa sezione converte i file **MP3 di Common Voice 26.0** già scaricati localmente
nel formato WAV 16 kHz / mono / 16-bit PCM, poi calcola le feature openWakeWord
(embedding shape `(N, 16, 96)`) salvandole in un file `.npy` tramite memory-mapping.

Il file risultante viene aggiunto a `feature_data_files` nel config YAML come sorgente
di negativi diretti, **in aggiunta** agli embedding ACAV100M già scaricati sopra.

> **Percorso corpus:** `1781719638236-cv-corpus-26.0-2026-06-12-it/cv-corpus-26.0-2026-06-12/it/clips`
> Modifica `CV_CORPUS_DIR` nella prima cella se la cartella si trova altrove.

In [ ]:
# === CONFIGURAZIONE Common Voice (corpus locale) ===
# Cartella contenente i file MP3 scaricati dal sito Mozilla Common Voice
CV_CORPUS_DIR  = r"1781719638236-cv-corpus-26.0-2026-06-12-it/cv-corpus-26.0-2026-06-12/it/clips"
output_dir_cv  = "./common_voice_it_wav"   # cartella WAV 16kHz di destinazione
CV_OUTPUT_NPY  = "./common_voice_it_features.npy"
CV_N_ORE       = 10   # ore massime di audio da convertire (None = tutto il corpus)
# =====================================================

mp3_files = sorted(Path(CV_CORPUS_DIR).glob("*.mp3"))
print(f"Trovati {len(mp3_files)} file MP3 in '{CV_CORPUS_DIR}'")

# Calcola quanti file corrispondono al limite di ore richiesto.
# mutagen legge la durata dall'header MP3 senza decodificare l'audio: molto più veloce.
if CV_N_ORE is not None:
    from mutagen.mp3 import MP3 as MutagenMP3
    secondi_accumulati = 0.0
    mp3_selezionati = []
    print(f"Selezione file fino a {CV_N_ORE} ore ({CV_N_ORE*3600:.0f} sec)...")
    for p in tqdm(mp3_files, desc="Lettura durate MP3"):
        try:
            durata = MutagenMP3(p).info.length
        except Exception:
            continue
        if secondi_accumulati + durata > CV_N_ORE * 3600:
            break
        mp3_selezionati.append(str(p))
        secondi_accumulati += durata
    print(f"Selezionati {len(mp3_selezionati)} file → {secondi_accumulati/3600:.2f} ore")
else:
    mp3_selezionati = [str(p) for p in mp3_files]
    print(f"CV_N_ORE = None → conversione dell'intero corpus ({len(mp3_selezionati)} file)")

if os.path.exists(output_dir_cv) and len(os.listdir(output_dir_cv)) >= len(mp3_selezionati) * 0.9:
    n_found = len(os.listdir(output_dir_cv))
    print(f"La cartella '{output_dir_cv}' esiste già e contiene {n_found} file WAV.")
    print("Conversione Common Voice saltata.")
else:
    print(f"Inizio conversione {len(mp3_selezionati)} file MP3 → WAV 16kHz mono 16-bit...")
    os.makedirs(output_dir_cv, exist_ok=True)

    # Usa HuggingFace datasets per decodifica + ricampionamento, identico alle altre celle
    cv_dataset = datasets.Dataset.from_dict({"audio": mp3_selezionati})
    cv_dataset = cv_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000, mono=True))

    saved_cv, skipped_cv = 0, 0

    for row in tqdm(cv_dataset, desc="Conversione Common Voice IT"):
        try:
            audio_array = row["audio"]["array"]

            if audio_array is None or len(audio_array) < 16000:  # scarta clip < 1 sec
                skipped_cv += 1
                continue

            nome_file = os.path.basename(row["audio"]["path"]).replace(".mp3", ".wav")
            scipy.io.wavfile.write(
                os.path.join(output_dir_cv, nome_file),
                16000,
                (audio_array * 32767).astype(np.int16)
            )
            saved_cv += 1

        except Exception:
            skipped_cv += 1
            continue  # salta clip corrotti o malformati

    print("\n" + "="*50)
    print("Conversione Common Voice completata.")
    print(f"File WAV salvati : {saved_cv}")
    print(f"File scartati   : {skipped_cv} (< 1 sec o corrotti)")
    print("="*50 + "\n")


In [ ]:
# Pre-calcolo delle feature openWakeWord dai WAV di Common Voice
# Il file .npy risultante ha shape (N, 16, 96) – identica al file ACAV100M

import openwakeword.data
import openwakeword.utils
from numpy.lib.format import open_memmap

if os.path.exists(CV_OUTPUT_NPY):
    n_righe = np.load(CV_OUTPUT_NPY, mmap_mode='r').shape[0]
    print(f"File '{CV_OUTPUT_NPY}' già presente ({n_righe} finestre). Pre-calcolo saltato.")
else:
    print(f"Inizio pre-calcolo feature da '{output_dir_cv}'...")

    # Carica il modello di embedding (frozen, identico a quello usato dal resto del training)
    F = openwakeword.utils.AudioFeatures(inference_framework="onnx")

    # Filtra i clip validi (stessa logica usata dal notebook manuale training_models.ipynb)
    cv_clips, cv_durations = openwakeword.data.filter_audio_paths(
        [output_dir_cv],
        min_length_secs=1.0,
        max_length_secs=60 * 30,
        duration_method="header"
    )

    # Applica il limite di ore anche in fase di embedding (coerente con la conversione)
    if CV_N_ORE is not None:
        secondi_acc = 0.0
        cv_clips_limitati, cv_dur_limitati = [], []
        for clip, dur in zip(cv_clips, cv_durations):
            if secondi_acc + dur > CV_N_ORE * 3600:
                break
            cv_clips_limitati.append(clip)
            cv_dur_limitati.append(dur)
            secondi_acc += dur
        cv_clips, cv_durations = cv_clips_limitati, cv_dur_limitati
        print(f"{len(cv_clips)} clip selezionati → {sum(cv_durations)/3600:.2f} ore (limite: {CV_N_ORE} ore)")
    else:
        print(f"{len(cv_clips)} clip validi trovati, ~{sum(cv_durations)/3600:.1f} ore totali")

    # Stima conservativa del numero di finestre (ogni clip da N sec → N/1.28 finestre)
    stima_finestre = int(sum(cv_durations) / 1.28)

    # Crea il file .npy con memory-mapping (permette array da centinaia di GB senza OOM)
    fp_cv = open_memmap(
        CV_OUTPUT_NPY, mode="w+",
        dtype=np.float16,
        shape=(stima_finestre, 16, 96)
    )

    row_counter = 0
    BATCH_SIZE_CV = 64  # clip per batch; riduci a 32 se si esaurisce la RAM

    pbar_feat = tqdm(range(0, len(cv_clips), BATCH_SIZE_CV), desc="Calcolo embedding CV")

    for i in pbar_feat:
        batch_paths = cv_clips[i : i + BATCH_SIZE_CV]
        batch_audio = []

        for path in batch_paths:
            try:
                _, data = scipy.io.wavfile.read(path)
                batch_audio.append(data)
            except Exception:
                continue

        if not batch_audio:
            continue

        # Stack a lunghezza fissa di 16000 camp. (1 sec) per uniformità
        stacked = openwakeword.data.stack_clips(
            batch_audio, clip_size=16000
        ).astype(np.int16)

        # Calcola gli embedding (output shape: (n_finestre, 16, 96))
        features = F.embed_clips(stacked, batch_size=BATCH_SIZE_CV)

        n = features.shape[0]
        if row_counter + n > stima_finestre:
            # Espandi il file se la stima era troppo bassa
            stima_finestre = row_counter + n + 10000
            fp_cv = open_memmap(
                CV_OUTPUT_NPY, mode="r+",
                dtype=np.float16,
                shape=(stima_finestre, 16, 96)
            )

        fp_cv[row_counter : row_counter + n] = features.astype(np.float16)
        row_counter += n
        pbar_feat.set_postfix({"finestre": row_counter})

    # Salva la versione finale troncata al numero reale di righe
    features_finali = np.array(fp_cv[:row_counter], dtype=np.float16)
    np.save(CV_OUTPUT_NPY, features_finali)
    del fp_cv

    print("\n" + "="*50)
    print("Pre-calcolo feature Common Voice completato.")
    print(f"Finestre totali salvate : {row_counter}")
    print(f"Shape finale            : ({row_counter}, 16, 96)")
    print(f"File output             : {CV_OUTPUT_NPY}")
    print("="*50 + "\n")


In [ ]:
output_dir = "./mit_rirs"

# Controllo intelligente: la cartella esiste E contiene già dei file?
if os.path.exists(output_dir) and len(os.listdir(output_dir)) > 0: #TO-DO iinserire limite
    print(f"La cartella '{output_dir}' esiste già e contiene i file audio.")
    print("Download saltato per risparmiare tempo.")
else:
    # Crea la cartella (senza dare errore se esiste già)
    os.makedirs(output_dir, exist_ok=True)
    print(f"Creazione cartella '{output_dir}' completata. Inizio il download...")
    
    # Carica il dataset da Hugging Face in streaming
    rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

    # Save clips to 16-bit PCM wav files
    for row in tqdm(rir_dataset, desc="Salvataggio RIRs"):
        name = row['audio']['path'].split('/')[-1]
        scipy.io.wavfile.write(
            os.path.join(output_dir, name), 
            16000, 
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    print("Salvataggio dei file .wav completato con successo!")

# Pulizia di eventuali cartelle temporanee residue
!rm -rf ./MIT_environmental_impulse_responses

In [ ]:
## Download noise and background audio

# Audioset Dataset (https://research.google.com/audioset/dataset/index.html)
# Download one part of the audioset .tar files, extract, and convert to 16khz
# For full-scale training, it's recommended to download the entire dataset from
# https://huggingface.co/datasets/agkphysics/AudioSet, and
# even potentially combine it with other background noise datasets (e.g., FSD50k, Freesound, etc.)

# === CONFIGURAZIONE DIRECTORY E PARAMETRI ===
# AudioSet
output_dir_audioset = "./audioset_16k"
temp_dir_audioset = "./audioset"
fname_audioset = "bal_train09.tar"
tar_path_audioset = f"{temp_dir_audioset}/{fname_audioset}"
link_audioset = f"https://huggingface.co/datasets/agkphysics/AudioSet/resolve/196c0900867eff791b8f4d4be57db277e9a5b131/{fname_audioset}"

# FMA
output_dir_fma = "./fma"
n_hours_fma = 10  
target_files_fma = n_hours_fma * 3600 // 30
# ============================================


# --- 1. AUDIOSET DATASET ---
if os.path.exists(output_dir_audioset) and len(os.listdir(output_dir_audioset)) > 0:#TO-DO iinserire limite
    print(f"La cartella '{output_dir_audioset}' esiste già e contiene i file audio.")
    print("Download e conversione AudioSet")
else:
    print("Inizio elaborazione AudioSet...")
    
    # Controllo e pulizia pre-download
    if os.path.exists(temp_dir_audioset):
        print(f"Trovata cartella temporanea '{temp_dir_audioset}' preesistente. Rimozione per forzare un download pulito...")
        shutil.rmtree(temp_dir_audioset, ignore_errors=True)
        
    os.makedirs(temp_dir_audioset, exist_ok=True)
    
    !wget -q -O {tar_path_audioset} {link_audioset} 
    !cd {temp_dir_audioset} && tar -xf {fname_audioset}

    os.makedirs(output_dir_audioset, exist_ok=True)

    # Convert audioset files to 16khz sample rate
    audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path(f"{temp_dir_audioset}/audio").glob("**/*.flac")]})
    audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
    
    for row in tqdm(audioset_dataset, desc="Conversione AudioSet"):
        name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
        scipy.io.wavfile.write(
            os.path.join(output_dir_audioset, name), 
            16000, 
            (row['audio']['array'] * 32767).astype(np.int16)
        )
        
    print("Elaborazione AudioSet completata con successo.")
    shutil.rmtree(temp_dir_audioset, ignore_errors=True)


# --- 2. FREE MUSIC ARCHIVE (FMA) DATASET ---
if os.path.exists(output_dir_fma) and len(os.listdir(output_dir_fma)) >= target_files_fma:
    print(f"La cartella '{output_dir_fma}' esiste già e contiene almeno {n_hours_fma} ora/e di audio.")
    print("Download FMA saltato")
else:
    print("Inizio download e conversione FMA...")
    os.makedirs(output_dir_fma, exist_ok=True)
    
    fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
    fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

    # Contatore e ciclo while per assicurarsi di avere esattamente 'target_files_fma' file validi
    saved_files = 0
    pbar = tqdm(total=target_files_fma, desc=f"Salvataggio FMA ({n_hours_fma} ora)")

    while saved_files < target_files_fma:
        try:
            row = next(fma_dataset)
            
            # Controllo di sicurezza per array vuoti o malformati
            if row.get('audio') is None or row['audio'].get('array') is None:
                continue
                
            name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
            
            # Salvataggio del file decodificato e quantizzato
            scipy.io.wavfile.write(
                os.path.join(output_dir_fma, name), 
                16000, 
                (row['audio']['array'] * 32767).astype(np.int16)
            )
            
            # Aggiornamento del contatore e della progress bar solo in caso di salvataggio riuscito
            saved_files += 1
            pbar.update(1)
            
        except StopIteration:
            print("\nAttenzione: Il dataset streaming è terminato prima di raggiungere il target.")
            break
        except Exception as e:
            # Ignora l'errore di dequantizzazione (o altri errori) e salta al brano successivo
            continue

    pbar.close()
    
    # Calcolo e stampa del riepilogo
    secondi_totali = saved_files * 30
    ore_effettive = secondi_totali / 3600
    
    print("\n" + "="*40)
    print("Elaborazione FMA completata.")
    print(f"File validi salvati: {saved_files} / {target_files_fma}")
    print(f"Ore totali scaricate: {ore_effettive:.2f} ore")
    print("="*40 + "\n")

In [ ]:
import os, shutil, zipfile
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

# === CONFIGURAZIONE ===
output_dir_esc50 = "./ESC-50_16k"
temp_dir_esc50   = "./esc50_temp"
zip_name_esc50   = "esc50.zip"
zip_path_esc50   = f"{temp_dir_esc50}/{zip_name_esc50}"
link_esc50       = "https://github.com/karolpiczak/ESC-50/archive/master.zip"
TARGET_SR        = 16000
# ======================

if os.path.exists(output_dir_esc50) and any(Path(output_dir_esc50).iterdir()):
    print(f"'{output_dir_esc50}' esiste già e contiene file. Salto.")
else:
    print("Inizio elaborazione ESC-50...")

    if os.path.exists(temp_dir_esc50):
        print(f"Rimozione cartella temporanea preesistente '{temp_dir_esc50}'...")
        shutil.rmtree(temp_dir_esc50, ignore_errors=True)

    os.makedirs(temp_dir_esc50, exist_ok=True)
    os.makedirs(output_dir_esc50, exist_ok=True)

    print("1. Download ESC-50 da GitHub (~600 MB)...")
    !wget -q --show-progress -O {zip_path_esc50} {link_esc50}

    print("2. Estrazione...")
    # Estrazione con zipfile nativo Python, nessuna dipendenza di sistema
    with zipfile.ZipFile(zip_path_esc50, 'r') as zip_ref:
        zip_ref.extractall(temp_dir_esc50)

    esc50_input_dir = Path(temp_dir_esc50) / "ESC-50-master" / "audio"
    wav_files = list(esc50_input_dir.glob("**/*.wav"))
    print(f"3. Conversione di {len(wav_files)} file a {TARGET_SR} Hz (16-bit PCM)...")

    saved, failed = 0, 0

    for wav_path in tqdm(wav_files, desc="Conversione ESC-50"):
        try:
            audio, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
            out_path  = os.path.join(output_dir_esc50, wav_path.name)
            sf.write(out_path, audio, TARGET_SR, subtype="PCM_16")
            saved += 1
        except Exception as e:
            failed += 1
            continue

    print("4. Rimozione file temporanei...")
    shutil.rmtree(temp_dir_esc50, ignore_errors=True)

    print("\n" + "="*40)
    print("Elaborazione ESC-50 completata!")
    print(f"Salvati: {saved} | Falliti: {failed} | Totale: {len(wav_files)}")
    print("="*40 + "\n")

In [ ]:
# --- DOWNLOAD, ESTRAZIONE E CONVERSIONE DI MUSAN --- TO-DO ristrutturare come gli altri
warnings.filterwarnings("ignore")
# Pulisce eventuali vecchi tentativi falliti
!rm -rf musan.tar.gz /content/musan_raw

print("1. Inizio download di MUSAN (~11 GB). Potrebbe richiedere 5-10 minuti...")
!wget -c --show-progress -O musan.tar.gz "https://openslr.elda.org/resources/17/musan.tar.gz"

print("\n2. Estrazione in corso... (Questa operazione richiede molta CPU, attendi...)")
!mkdir -p ./musan_raw
!tar -xf musan.tar.gz -C ./musan_raw

# Eliminiamo lo zip da 11GB per non esaurire lo spazio su disco!
!rm musan.tar.gz
print("Estrazione completata e archivio eliminato per liberare spazio.")

# Definisci le cartelle
musan_input_dir = "./musan_raw/musan"
dataset_output_dir = "./musan_16k"
os.makedirs(dataset_output_dir, exist_ok=True)

print("\n3. Preparazione dei file audio in Hugging Face...")
# Troviamo tutti i file wav nelle varie cartelle (noise, speech, music)
file_paths = [str(i) for i in Path(musan_input_dir).glob("**/*.wav")]
print(f"Trovati {len(file_paths)} file. Forzo il ricampionamento...")

nuovo_dataset = datasets.Dataset.from_dict({"audio": file_paths})
nuovo_dataset = nuovo_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

file_saltati = 0

print("\n4. Inizio conversione in 16-bit PCM a 16kHz...")
for i in tqdm(range(len(nuovo_dataset))):
    try:
        row = nuovo_dataset[i]

        # Usiamo un nome progressivo sicuro per evitare che file omonimi si sovrascrivano
        nome_file = f"musan_{i:06d}.wav"

        scipy.io.wavfile.write(
            os.path.join(dataset_output_dir, nome_file),
            16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    except Exception:
        file_saltati += 1

print("\n Conversione completata!")
if file_saltati > 0:
    print(f"Ignorati {file_saltati} file corrotti o illeggibili.")

#Pulizia finale della cartella grezza per svuotare il disco di Colab
print("\n5. Pulizia dei file grezzi originali...")
!rm -rf ./musan_raw

print("Finito, I tuoi file sono in /content/musan_16k")

In [ ]:
# --- DOWNLOAD E PREPARAZIONE OPENSLR 26 SIMULATES RIRS ---
import os
from pathlib import Path

# === CONFIGURAZIONE ===
output_dir_openslr = "./openslr26_rirs"
zip_name = "sim_rir_16k.zip"
link = "https://openslr.elda.org/resources/26/sim_rir_16k.zip"

# === ESECUZIONE ===
if os.path.exists(output_dir_openslr) and len(os.listdir(output_dir_openslr)) > 0:
    print("Dataset già presente.")
else:
    # 1. Download tramite wget
    print("1. Inizio download di OpenSLR 26 RIRs...")
    !wget -q --show-progress -O {zip_name} {link}
    
    # 2. Estrazione tramite unzip
    print("2. Estrazione in corso...")
    os.makedirs(output_dir_openslr, exist_ok=True)
    !unzip -q {zip_name} -d {output_dir_openslr}
    
    # 3. Pulizia
    print("3. Rimozione archivio zip...")
    os.remove(zip_name)
    
    # Verifica
    file_paths = list(Path(output_dir_openslr).glob("**/*.wav"))
    print(f"Operazione completata: {len(file_paths)} file estratti.")

In [ ]:
# Percorso della tua cartella principale
base_dir = Path("./openslr26_rirs")

print(f"Appiattimento della cartella {base_dir} in corso...")

# Trova tutti i file .wav in qualsiasi sottocartella
for file_path in base_dir.rglob("*.wav"):
    # Se il file è già nella cartella base, saltalo
    if file_path.parent == base_dir:
        continue

    # Crea un nome univoco basato sulla sottocartella per evitare conflitti
    # Esempio: sottocartella/audio.wav -> sottocartella_audio.wav
    new_name = f"{file_path.parent.name}_{file_path.name}"
    target_path = base_dir / new_name

    # Sposta il file
    shutil.move(str(file_path), str(target_path))
    print(f"Spostato: {file_path.name} -> {new_name}")

# Elimina le sottocartelle ora vuote
for subfolder in base_dir.iterdir():
    if subfolder.is_dir():
        shutil.rmtree(subfolder)

print("Operazione completata! Ora tutti i RIR sono nella cartella principale.")

In [ ]:
# --- DOWNLOAD E CONVERSIONE DI URBANSOUND8K --- TO-DO ristrutturare come altri
warnings.filterwarnings("ignore")

print("1. Inizio download di UrbanSound8K tramite Hugging Face...")
us8k_out = "./urbansound_16k"
os.makedirs(us8k_out, exist_ok=True)

# Caricamento del dataset
us_dataset = datasets.load_dataset("danavery/urbansound8K", split="train")
us_dataset = us_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))

print(f"2. Inizio conversione di {len(us_dataset)} file in 16-bit PCM a 16kHz...")
for i, row in enumerate(tqdm(us_dataset, desc="Conversione UrbanSound8K")):
    try:
        # Creiamo un nome univoco usando l'indice e la classe del rumore
        name = f"us8k_{i:04d}_{row['class']}.wav"
        wav.write(
            os.path.join(us8k_out, name),
            16000,
            (row['audio']['array'] * 32767).astype(np.int16)
        )
    except Exception:
        pass # Ignora file rari che potrebbero essere corrotti

print("✅ UrbanSound8K pronto e convertito!")

# Define Training Configuration

For automated model training openWakeWord uses a specially designed training script and a [YAML](https://yaml.org/) configuration file that defines all of the information required for training a new wake word/phrase detection model.

It is strongly recommended that you review [the example config file](../examples/custom_model.yml), as each value is fully documented there. For the purposes of this notebook, we'll read in the YAML file to modify certain configuration parameters before saving a new YAML file for training our example model. Specifically:

- We'll train a detection model for the phrase "hey sebastian"
- We'll only generate 5,000 positive and negative examples (to save on time for this example)
- We'll only generate 1,000 validation positive and negative examples for early stopping (again to save time)
- The model will only be trained for 10,000 steps (larger datasets will benefit from longer training)
- We'll reduce the target metrics to account for the small dataset size and limited training.

On the topic of target metrics, there are *not* specific guidelines about what these metrics should be in practice, and you will need to conduct testing in your target deployment environment to establish good thresholds. However, from very limited testing the default values in the config file (accuracy >= 0.7, recall >= 0.5, false-positive rate <= 0.2 per hour) seem to produce models with reasonable performance.


In [5]:
# Load default YAML config file for training
config = yaml.load(open("openwakeword/examples/custom_model.yml", 'r').read(), yaml.Loader)


# Modify values in the config and save a new version
#config["tts_batch_size"] = 64
config["target_phrase"] = ["picsi"]
config["custom_negative_phrases"] = ["Pizza", "Pixel", "Pyrex", "Taxi", "Dixit", "Dixie", "Pasticcio", "Bixby", "Pipsi", "Mix"]

#config["length_scales"] = [0.75, 0.9, 1.0, 1.2] TO-DO in train.py, a cui serve il file di config.yaml, non espone configurazioni
#config["noise_scales"] = [0.333, 0.667, 0.98]   # per questi parametri del piper samples generator anche se li usa.
#config["noise_scale_ws"] = [0.333, 0.667, 0.98]

config["model_name"] = config["target_phrase"][0].replace(" ", "_")

config["n_samples"] = 20000
config["n_samples_val"] = 5000
config["steps"] = 50000
config["max_negative_weight"] = 300
config["layer_size"] = 32

config["target_accuracy"] = 0.75
config["target_recall"] = 0.65
config["target_false_positives_per_hour"] = 0.2



config["output_dir"] = "./my_custom_model"

config["background_paths"] = [
    './audioset_16k',
    './fma',
    './ESC-50_16k',
    './musan_16k',
    './urbansound_16k'
]

config["rir_paths"] = [
    './openslr26_rirs',
    './mit_rirs'
]

config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}


#config["custom_data_dir"] = "./miei_audio_reali"

with open('my_model.yaml', 'w') as file:
    documents = yaml.dump(config, file)

# Train the Model

With the data downloaded and training configuration set, we can now start training the model. We'll do this in parts to better illustrate the sequence, but you can also execute every step at once for a fully automated process.

In [2]:
# Step 1: Generate synthetic clips
# For the number of clips we are using, this should take ~10 minutes on a free Google Colab instance with a T4 GPU
# If generation fails, you can simply run this command again as it will continue generating until the
# number of files meets the targets specified in the config file

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --generate_clips

INFO:root:##################################################
Generating positive clips for training
##################################################
INFO:root:##################################################
Generating positive clips for testing
##################################################
INFO:root:##################################################
Generating negative clips for training
##################################################
INFO:root:##################################################
Generating negative clips for testing
##################################################
DEBUG:generate_samples:Loading /home/franc/openwakeword_project/piper-sample-generator/models/en_US-libritts_r-medium.pt
INFO:generate_samples:Successfully loaded the model
DEBUG:generate_samples:CUDA available, using GPU
DEBUG:generate_samples:Batch 1/714 complete
DEBUG:generate_samples:Batch 2/714 complete
DEBUG:generate_samples:Batch 3/714 complete
DEBUG:generate_samples:Batch 4/714 complete


In [ ]:
import os
import numpy as np
import scipy.io.wavfile as wav
from glob import glob

# Percorso base
cartella_base = "./my_custom_model/picsi"

# Solo le cartelle con gli esempi positivi
sottocartelle = ["positive_test", "positive_train"]

# Valore fisso per ridurre il volume (es. 0.5 dimezza, 0.7 riduce del 30%)
fattore_riduzione_positivi = 0.25 # 0.25 riduce di 3/4

print("\n" + "="*55)
print(" INIZIO TRATTAMENTO DEL VOLUME AUDIO (SOLO POSITIVI)")
print("="*55)

for sottocartella in sottocartelle:
    percorso_cartella = os.path.join(cartella_base, sottocartella)
    files = glob(f"{percorso_cartella}/*.wav")
    
    n_files = len(files)
    if n_files == 0:
        print(f"\nCartella: {sottocartella} -> Nessun file .wav trovato. Salto.")
        continue
        
    print(f"\nCartella: {sottocartella} | File totali: {n_files}")
    print(f"  -> Applico riduzione fissa ({fattore_riduzione_positivi}x) a {n_files} file...")
    
    file_processati = 0
    for file in files:
        file_processati += 1
        
        try:
            rate, data = scipy.io.wavfile.read(file)
            # Applica la riduzione e formatta come intero a 16-bit
            dati_modificati = (data * fattore_riduzione_positivi).astype(np.int16)
            scipy.io.wavfile.write(file, rate, dati_modificati)
        except Exception as e:
            nome_file = os.path.basename(file)
            print(f"\n   Errore su {nome_file}: {e}")

        # Feedback visivo sulla stessa riga
        if file_processati % 50 == 0 or file_processati == n_files:
            print(f"     Progresso: [{file_processati}/{n_files}] file elaborati...", end="\r")
    
    print(f"\n {sottocartella} completata!")

print("\n" + "="*55)
print(" OPERAZIONE COMPLETATA CON SUCCESSO! ")
print("="*55 + "\n")

In [7]:
# Step 2: Augment the generated clips

!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --augment_clips

^C
Traceback (most recent call last):
  File "/home/franc/openwakeword_project/openwakeword/openwakeword/train.py", line 18, in <module>
    import openwakeword
  File "/home/franc/openwakeword_project/openwakeword/openwakeword/__init__.py", line 4, in <module>
    from openwakeword.custom_verifier_model import train_custom_verifier
  File "/home/franc/openwakeword_project/openwakeword/openwakeword/custom_verifier_model.py", line 23, in <module>
    from sklearn.linear_model import LogisticRegression
  File "/home/franc/openwakeword_project/venv/lib/python3.10/site-packages/sklearn/__init__.py", line 73, in <module>
    from .base import clone  # noqa: E402
  File "/home/franc/openwakeword_project/venv/lib/python3.10/site-packages/sklearn/base.py", line 19, in <module>
    from .utils._metadata_requests import _MetadataRequester, _routing_enabled
  File "/home/franc/openwakeword_project/venv/lib/python3.10/site-packages/sklearn/utils/__init__.py", line 9, in <module>
    from ._chunkin

In [ ]:
# Step 3: Train model
!{sys.executable} openwakeword/openwakeword/train.py --training_config my_model.yaml --train_model

In [ ]:
# Step 4 (Optional): On Google Colab, sometimes the .tflite model isn't saved correctly
# If so, run this cell to retry

# Manually save to tflite as this doesn't work right in colab
def convert_onnx_to_tflite(onnx_model_path, output_path):
    """Converts an ONNX version of an openwakeword model to the Tensorflow tflite format."""
    # imports
    import onnx
    import logging
    import tempfile
    from onnx_tf.backend import prepare
    import tensorflow as tf

    # Convert to tflite from onnx model
    onnx_model = onnx.load(onnx_model_path)
    tf_rep = prepare(onnx_model, device="CPU")
    with tempfile.TemporaryDirectory() as tmp_dir:
        tf_rep.export_graph(os.path.join(tmp_dir, "tf_model"))
        converter = tf.lite.TFLiteConverter.from_saved_model(os.path.join(tmp_dir, "tf_model"))
        tflite_model = converter.convert()

        logging.info(f"####\nSaving tflite mode to '{output_path}'")
        with open(output_path, 'wb') as f:
            f.write(tflite_model)

    return None

convert_onnx_to_tflite(f"my_custom_model/{config['model_name']}.onnx", f"my_custom_model/{config['model_name']}.tflite")


After the model finishes training, the auto training script will automatically convert it to ONNX and tflite versions, saving them as `my_custom_model/<model_name>.onnx/tflite` in the present working directory, where `<model_name>` is defined in the YAML training config file. Either version can be used as normal with `openwakeword`. I recommend testing them with the [`detect_from_microphone.py`](https://github.com/dscripka/openWakeWord/blob/main/examples/detect_from_microphone.py) example script to see how the model performs!